Big picture — what is fine-tuning?

We are not teaching language from zero.

BERT already knows:

grammar
sentence structure,
word relationships,
context,
language meaning

because it read billions of words during pretraining.

So instead of building a new language brain:

We take BERT and slightly adapt it.

This is called:

Fine-tuning




 Compare with ResNet transfer learning

 This is almost identical to Week 2.

| Vision             | NLP                 |
| ------------------ | ------------------- |
| ResNet             | BERT                |
| ImageNet knowledge | Language knowledge  |
| Replace FC layer   | Add classifier head |
| Freeze/fine-tune   | Fine-tune BERT      |
| Small LR           | Small LR            |




In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader

# Load SST-2
dataset   = load_dataset('glue', 'sst2')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print(dataset)
print(dataset['train'][0])   # {'sentence': '...', 'label': 1, 'idx': 0}


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)

# Set format for PyTorch
tokenized.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

train_loader = DataLoader(tokenized['train'],      batch_size=32, shuffle=True)
val_loader   = DataLoader(tokenized['validation'], batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")


In [ ]:
batch = next(iter(train_loader))
print("input_ids shape:      ", batch['input_ids'].shape)       # [32, 128]
print("attention_mask shape: ", batch['attention_mask'].shape)  # [32, 128]
print("labels shape:         ", batch['label'].shape)           # [32]
print("label values:         ", batch['label'][:8])             # 0s and 1s'

In [ ]:
# For fast iteration during development — use a subset
from torch.utils.data import Subset
import random

train_idx = random.sample(range(len(tokenized['train'])), 4000)
val_idx   = random.sample(range(len(tokenized['validation'])), 500)

fast_train = DataLoader(Subset(tokenized['train'], train_idx),
                        batch_size=32, shuffle=True)
fast_val   = DataLoader(Subset(tokenized['validation'], val_idx),
                        batch_size=32, shuffle=False)
print(f"Fast train: {len(fast_train)} batches | Fast val: {len(fast_val)} batches")

Big picture of this whole section

You are building the NLP training pipeline:

SST-2 text
↓
Tokenizer
↓
input_ids + masks
↓
Tensor conversion
↓
Batches
↓
DataLoader
↓
Ready for BERT fine-tuning

No learning happens yet.

This is:

Data preparation stage for NLP training.

Exactly analogous to:

Titanic preprocessing
CIFAR transforms + DataLoader
One-line memory shortcut

Remember:

SST-2 provides labelled sentiment text, tokenizer converts text into BERT-readable numbers, and DataLoader batches everything so BERT can train efficiently.

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class BertSentimentClassifier(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_classes=2, dropout=0.3):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size   # 768 for bert-base

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids,
                               attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        logits     = self.classifier(cls_output)
        return logits

model = BertSentimentClassifier()
print(model)
total = sum(p.numel() for p in model.parameters())
print(f"\nTotal params: {total:,}")   # ~109M

In [ ]:
from transformers import AutoModelForSequenceClassification

# One line — BERT + classification head, all wired up
model_hf = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)
print(model_hf.classifier)   # see the built-in head

This step is very similar to what you did with ResNet transfer learning in Week 2.

Same pattern:

Pretrained backbone + new task-specific head

Only difference:

ResNet → understands images
BERT → understands language

So don't think of this as a new idea. It is transfer learning again, but for text.

Big picture

You want to classify sentiment:

Input:
"This movie was amazing"

↓

BERT understands the sentence

↓

Small classifier decides sentiment

↓

Output: Positive / Negative

Final mental model
This whole model is:
Pretrained language expert+small sentiment decision layer

or:

Sentence→ BERT understands→ CLS summarizes→ Head predicts sentiment
Same transfer learning idea you already learned with ResNet — just moved from images → language.

In [ ]:
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = BertSentimentClassifier().to(device)

# AdamW — Adam with weight decay fix, standard for transformers
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

# Linear warmup scheduler — standard for BERT fine-tuning
total_steps  = len(fast_train) * 3   # 3 epochs
warmup_steps = total_steps // 10
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

def run_epoch(model, loader, optimizer, criterion, scheduler, device, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return total_loss / len(loader), acc

print("Training BERT — this will take a few minutes per epoch on CPU...")
for epoch in range(3):
    tr_loss, tr_acc = run_epoch(model, fast_train, optimizer,
                                criterion, scheduler, device, train=True)
    vl_loss, vl_acc = run_epoch(model, fast_val,   optimizer,
                                criterion, scheduler, device, train=False)
    print(f"Epoch {epoch+1} | train loss {tr_loss:.4f} acc {tr_acc:.3f} "
          f"| val loss {vl_loss:.4f} acc {vl_acc:.3f}")

Big picture first

You already have:

Text → Tokenizer → BERT → [CLS] → Classifier head

Now this section teaches:

How does BERT actually learn sentiment?

The answer:

Pass sentences through BERT

1. Compare prediction vs true label
2. Calculate error (loss)
3. Backpropagate gradients
4. Update weights slightly
5. Repeat many times

Same training loop as Week 2 — only the model is now BERT.

Here we use AdamW.

Why AdamW?

Think:

Adam = smart gradient descent.

AdamW = Adam + cleaner regularisation.

Transformers almost always use:

AdamW
tiny learning rates

because BERT is pretrained and delicate.

----------
Why not lr=0.001 like CNN?

Because:

CNN from scratch:

random weights → learn aggressively

BERT:

already excellent → only slight adjustment

Large LR would damage pretrained knowledge.

This is called:

catastrophic forgetting.

Connection to Week 2

This should now feel familiar.

| Week 2 CNN                 | Week 3 BERT              |
| -------------------------- | ------------------------ |
| Images                     | Text                     |
| ConvNet                    | Transformer              |
| CrossEntropy               | CrossEntropy             |
| Adam                       | AdamW                    |
| Forward                    | Forward                  |
| Backward                   | Backward                 |
| Scheduler optional         | Scheduler standard       |
| Transfer learning (ResNet) | Transfer learning (BERT) |

Short memory summary

This section teaches four new ideas:

1. AdamW
Transformer-friendly optimizer.
2. Tiny LR (2e-5)
Protect pretrained knowledge.
3. Warmup + scheduler
Slow start, gradual decay.
4. Gradient clipping
Prevents unstable training.

Everything else is the same PyTorch loop you already know.

In [ ]:
def predict_sentiment(texts, model, tokenizer, device):
    model.eval()
    inputs = tokenizer(
        texts, return_tensors='pt',
        padding=True, truncation=True, max_length=128
    )
    input_ids      = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        probs  = torch.softmax(logits, dim=1)
        preds  = torch.argmax(probs, dim=1)

    labels = ['negative', 'positive']
    for text, pred, prob in zip(texts, preds, probs):
        sentiment   = labels[pred.item()]
        confidence  = prob[pred.item()].item()
        print(f"[{sentiment:8s} {confidence:.1%}]  {text}")

# Try your own sentences
test_sentences = [
    "This is the best movie I have seen in years.",
    "Absolutely terrible. I want my money back.",
    "It was okay I guess, nothing special.",
    "A masterpiece of modern cinema.",
    "The acting was wooden and the plot made no sense.",
    "I fell asleep halfway through.",            # edge case
    "Not bad, but not great either.",            # neutral — how does it handle this?
]

predict_sentiment(test_sentences, model, tokenizer, device)

In [ ]:
torch.save(model.state_dict(), 'bert_sentiment.pth')

# To reload later:
# model = BertSentimentClassifier()
# model.load_state_dict(torch.load('bert_sentiment.pth', map_location='cpu'))
# model.eval()

Short memory summary
This section teaches inference.
New ideas:


Softmax
Raw scores → probabilities.


Confidence
How strongly model prefers a class.


Neutral limitation
Binary classifiers must choose.


Model saving
Preserve learned weights.


Everything else is the same BERT pipeline you already learned:
Text→ Tokenizer→ BERT→ CLS→ Classifier→ Prediction